In [11]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import (
    PyPDFLoader,
    Docx2txtLoader,
    TextLoader,
)
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

CHUNK_SIZE = 500
CHUNK_OVERLAP = 50
EMBED_MODEL = "all-MiniLM-L6-v2"
GROQ_MODEL = "openai/gpt-oss-20b"
TOP_K = 6

print("Imports OK")

Imports OK


In [3]:
def load_document(file_path):
    """Load a PDF, DOCX, or TXT file and return a list of documents."""
    ext = os.path.splitext(file_path)[1].lower()

    if ext == ".pdf":
        loader = PyPDFLoader(file_path)
    elif ext == ".docx":
        loader = Docx2txtLoader(file_path)
    elif ext == ".txt":
        loader = TextLoader(file_path, encoding="utf-8")
    else:
        raise ValueError(f"Unsupported file type: {ext}")

    docs = loader.load()

    for d in docs:
        d.metadata["source_file"] = os.path.basename(file_path)

    return docs


all_docs = []
for path in ["test.pdf", "test.docx", "test.txt"]:
    if os.path.exists(path):
        docs = load_document(path)
        all_docs.extend(docs)
        print(f"✅ Loaded {len(docs)} pages from {path}")
    else:
        print(f"⚠️  Skipped {path} (file not found)")

print(f"\nTotal pages across all documents: {len(all_docs)}")

✅ Loaded 11 pages from test.pdf
✅ Loaded 1 pages from test.docx
✅ Loaded 1 pages from test.txt

Total pages across all documents: 13


In [4]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = splitter.split_documents(all_docs)

for i, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = i

print(f"Created {len(chunks)} chunks")

print("\nSample metadata (first chunk):")
for k, v in chunks[0].metadata.items():
    print(f"  {k}: {v}")

print("\nSource distribution:")
sources = {}
for c in chunks:
    src = c.metadata.get("source_file", "unknown")
    sources[src] = sources.get(src, 0) + 1
for src, count in sources.items():
    print(f"  {src}: {count} chunks")

Created 133 chunks

Sample metadata (first chunk):
  producer: pikepdf 8.15.1
  creator: arXiv GenPDF (tex2pdf:0d14211)
  creationdate: 
  author: Yeva Gevorgyan
  doi: https://doi.org/10.48550/arXiv.2610.02026
  license: http://arxiv.org/licenses/nonexclusive-distrib/1.0/
  ptex.fullbanner: This is pdfTeX, Version 3.141592653-2.6-1.40.28 (TeX Live 2025) kpathsea version 6.4.1
  title: Interpreting effective homogeneous rheologies through a local differential map with application to the Moon
  trapped: /False
  arxivid: https://arxiv.org/abs/2610.02026v1
  source: test.pdf
  total_pages: 11
  page: 0
  page_label: 1
  source_file: test.pdf
  chunk_id: 0

Source distribution:
  test.pdf: 118 chunks
  test.docx: 14 chunks
  test.txt: 1 chunks


In [5]:
import shutil
from langchain_community.vectorstores import Chroma

if os.path.exists("./chroma_db_week2"):
    shutil.rmtree("./chroma_db_week2")
    print("🗑️  Deleted old chroma_db_week2")

embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={"device": "cpu"}
)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db_week2"
)

print(f"✅ Vector store ready: {vectorstore._collection.count()} vectors")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Vector store ready: 133 vectors


In [12]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": TOP_K}
)

def format_docs_with_citations(docs):
    """Convert retrieved docs into numbered context with source labels."""
    formatted = []
    for i, doc in enumerate(docs, start=1):
        source = doc.metadata.get("source_file", "unknown")
        page = doc.metadata.get("page_label", doc.metadata.get("page", "?"))
        label = f"[{i}] ({source}, page {page})"
        formatted.append(f"{label}\n{doc.page_content}")
    return "\n\n".join(formatted)

# Quick test
test_docs = retriever.invoke("What is this document about?")
print(f"Retrieved {len(test_docs)} chunks\n")
for i, doc in enumerate(test_docs, start=1):
    src = doc.metadata.get("source_file", "?")
    page = doc.metadata.get("page_label", doc.metadata.get("page", "?"))
    print(f"[{i}] {src} — page {page}")
    print(f"    {doc.page_content[:120].strip()}...")
    print()

Retrieved 6 chunks

[1] test.pdf — page 2
    thespectraldata(𝑘 𝐸,𝑠𝑖,𝑟𝑖)tothegeneralizedVoigtparameters.We
differentiate this map in Section 4.
MNRAS000, 1–11 (2026)...

[2] test.pdf — page 11
    Tobie G., Mocquet A., Sotin C., 2005, Icarus, 177, 534
Tromp J., Mitrovica J. X., 1999, Geophysical Journal Internationa...

[3] test.pdf — page 3
    is smooth with nonsingular Jacobian𝐽𝑤 =𝜕𝑢/𝜕𝑤, so its inverse
I:𝑢↦→𝑤is a local diffeomorphism at𝑢 0.
Throughout the paper...

[4] test.docx — page ?
    6. Suggested Page Layout

Page 1: Executive Summary (KPIs + revenue trend + top category). Page 2: Product Deep-Dive. Pa...

[5] test.pdf — page 7
    in Section 6.3.
The large value𝜎1 =91reflects a near-degenerate pair of Voigt
elements with times near1.2–1.3Myr rather...

[6] test.pdf — page 11
    Interpreting effective rheologies11
ygevorgyan/local-differential-mapandarchivedonZenodo
with a citable DOI (Gevorgyan 2...



In [7]:
template = """You are a helpful research assistant that answers questions strictly from the provided context.

RULES:
1. Answer ONLY using the information in the context below.
2. Every factual claim must end with a citation like [1] or [2].
3. If multiple sources support a claim, cite all: [1][3].
4. If the answer is NOT in the context, respond exactly: "I don't have enough information to answer that."

Context:
{context}

Question: {question}

Answer (with inline citations):"""

prompt = PromptTemplate.from_template(template)
print("Prompt template ready")

Prompt template ready


In [14]:
llm = ChatGroq(
    model_name=GROQ_MODEL,
    temperature=0,
    groq_api_key=os.getenv("GROQ_API_KEY")
)

def answer_with_sources(question):
    docs = retriever.invoke(question)
    context = format_docs_with_citations(docs)

    # Extract document titles as a hint
    titles = set()
    for d in docs:
        t = d.metadata.get("title")
        if t:
            titles.add(t)
    title_hint = ""
    if titles:
        title_hint = f"\nNote: The retrieved context comes from these documents: {'; '.join(titles)}.\n"

    chain = prompt | llm | StrOutputParser()
    answer = chain.invoke({
        "context": context + title_hint,
        "question": question
    })
    return answer, docs

print("RAG chain with citations ready")

RAG chain with citations ready


In [9]:
def ask(question):
    print(f"❓ QUESTION: {question}\n")
    answer, sources = answer_with_sources(question)
    print(f"💡 ANSWER:\n{answer}\n")
    print("─" * 70)
    print("📚 SOURCES:")
    for i, doc in enumerate(sources, start=1):
        src = doc.metadata.get("source_file", "unknown")
        page = doc.metadata.get("page_label", doc.metadata.get("page", "?"))
        preview = doc.page_content[:140].replace("\n", " ").strip()
        print(f"  [{i}] {src} (page {page})")
        print(f"      {preview}...")
    print("=" * 70)

print("Ready. Run: ask('your question')")

Ready. Run: ask('your question')


In [15]:
ask("What is the main topic of the PDF document?")

❓ QUESTION: What is the main topic of the PDF document?

💡 ANSWER:
The PDF is focused on **interpreting effective homogeneous rheologies through a local differential map applied to the Moon**. It presents a baseline model of the Moon’s interior layers (solid inner core, liquid outer core, low‑viscosity zone, upper mantle, and crust) with their densities, rigidities, and viscosities, and discusses spectral analysis and toy‑model approaches to understand lunar rheology. [3][4][5][6]

──────────────────────────────────────────────────────────────────────
📚 SOURCES:
  [1] test.docx (page ?)
      6. Suggested Page Layout  Page 1: Executive Summary (KPIs + revenue trend + top category). Page 2: Product Deep-Dive. Page 3: Geography. Pag...
  [2] test.docx (page ?)
      Dashboard 3: Territory / Geographic Sales  Filled map or Shape map visual: Total Revenue by Territory[Country]  Bar chart: Revenue by Territ...
  [3] test.pdf (page 2)
      thespectraldata(𝑘 𝐸,𝑠𝑖,𝑟𝑖)tothegeneralizedVoigtpara

In [13]:
# Ask only about the PDF
retriever_pdf = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4, "filter": {"source_file": "test.pdf"}}
)

def ask_pdf(question):
    docs = retriever_pdf.invoke(question)
    context = format_docs_with_citations(docs)
    chain = prompt | llm | StrOutputParser()
    answer = chain.invoke({"context": context, "question": question})
    print(f"❓ {question}\n")
    print(f"💡 {answer}\n")
    print("─" * 70)
    for i, doc in enumerate(docs, start=1):
        page = doc.metadata.get("page_label", "?")
        print(f"  [{i}] {doc.metadata.get('source_file')} (page {page})")
    print("=" * 70)

ask_pdf("What celestial body does this paper study?")

❓ What celestial body does this paper study?

💡 The paper studies the Moon. [4]

──────────────────────────────────────────────────────────────────────
  [1] test.pdf (page 11)
  [2] test.pdf (page 11)
  [3] test.pdf (page 11)
  [4] test.pdf (page 8)


In [16]:
ask("What is the capital of France?")

❓ QUESTION: What is the capital of France?

💡 ANSWER:
I don't have enough information to answer that.

──────────────────────────────────────────────────────────────────────
📚 SOURCES:
  [1] test.docx (page ?)
      Dashboard 3: Territory / Geographic Sales  Filled map or Shape map visual: Total Revenue by Territory[Country]  Bar chart: Revenue by Territ...
  [2] test.docx (page ?)
      TerritoryKey  Returns_Cleaned  Calendar_Cleaned  ReturnDate ↔ Date  Concept: this is a 'star schema' — one or more fact tables (transactiona...
  [3] test.pdf (page 1)
      nine-poleresponseoverthisinterval.Finiteperturbationsconfirmthetangentpredictiontowithinafewpercent,withthevalidity radius limited by a near...
  [4] test.pdf (page 8)
      Crust 1737.15 2735.00 1510 23 Table2.Robustnessofthemainfull-mapdiagnosticsunderselectedchangesofthebaselineviscosities.Inallcases,thespectr...
  [5] test.docx (page ?)
      Age & FullName added  Territory.csv  Dimension  Region/Country/Continent  Calendar_Cle